In [ ]:
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer

torch.set_float32_matmul_precision('high')
device = torch.device('cuda')

with open('config.yaml') as f:
    config = yaml.safe_load(f)

tokenizer = Tokenizer.from_file('tokenizer/tokenizer.json')
vocab_size = tokenizer.get_vocab_size()
mcfg = config['model']


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.0)
        return self.proj(out.transpose(1, 2).contiguous().view(B, T, C))


class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down(F.gelu(self.up(x)))


class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout, use_checkpoint):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.use_checkpoint = use_checkpoint

    def forward(self, x):
        if self.use_checkpoint and self.training:
            return checkpoint(self._forward, x, use_reentrant=False)
        return self._forward(x)

    def _forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_length, dropout, use_checkpoint):
        super().__init__()
        self.context_length = context_length
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(context_length, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(d_model, n_heads, d_ff, dropout, use_checkpoint)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.tok_emb.weight = self.lm_head.weight

    def forward(self, idx):
        B, T = idx.shape
        x = self.drop(self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device)))
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.ln_f(x))


model = GPT(
    vocab_size=vocab_size,
    d_model=mcfg['d_model'],
    n_heads=mcfg['n_heads'],
    n_layers=mcfg['n_layers'],
    d_ff=mcfg['d_ff'],
    context_length=mcfg['context_length'],
    dropout=0.0,
    use_checkpoint=False,
).to(device)

ckpt = torch.load('checkpoints/best.pt', weights_only=False, map_location=device) # change this to other checkpoints such as step_4000.pt (intervals of 2000 are logged) to try different variants
model.load_state_dict(ckpt['model'])
model.eval()
print(f"Loaded checkpoint from step {ckpt.get('step', '?')}")

In [ ]:
@torch.no_grad()
def generate(prompt, max_tokens=200, temperature=0.8, top_k=50):
    ids = tokenizer.encode(prompt).ids
    ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    for _ in range(max_tokens):
        idx = ids[:, -mcfg['context_length']:]
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            logits = model(idx)[:, -1] / temperature
        if top_k > 0:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float('-inf')
        ids = torch.cat([ids, torch.multinomial(F.softmax(logits, dim=-1), 1)], dim=1)
    return tokenizer.decode(ids[0].tolist())

In [ ]:
prompts = [
    'In a distant galaxy,',
    'def fibonacci(n):',
    'Once upon a time',
]

for p in prompts:
    print(f'--- {p} ---')
    print(generate(p))
    print()